In [16]:
import os
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import json
from pyspark.ml.functions import vector_to_array
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, Bucketizer
import matplotlib.pyplot as plt
import math
import pyspark
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType

In [17]:
pyspark_home = os.path.dirname(pyspark.__file__)

os.environ['SPARK_HOME'] = pyspark_home
os.environ['PATH'] += os.pathsep + os.path.join(pyspark_home, 'bin')
# Manually point to your Java and Hadoop locations
os.environ['JAVA_HOME'] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ['HADOOP_HOME'] = r"C:\hadoop"

# Force the 'bin' folders into the session path
os.environ['PATH'] += os.pathsep + os.path.join(os.environ['JAVA_HOME'], 'bin')
os.environ['PATH'] += os.pathsep + os.path.join(os.environ['HADOOP_HOME'], 'bin')

# Tell Spark exactly which python to use
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

if "spark" in globals():
    spark.stop()
spark = SparkSession.builder.appName("CarFeatureEngineering").config("spark.sql.ansi.enabled", "false").getOrCreate()
sc = spark.sparkContext

In [18]:
df = spark.read.csv("data/interim/used_cars_cleaned.csv", header=True, inferSchema=True)

# Print the top 10 rows
df.show(5)

feature_engineering_logs = []
total_count=df.count()
print(f"Total records: {total_count}")

+--------------------+----------------+-------+-----------------+--------+------------+---------+------+-------+------------+------------+-------+---------+----------+----+
|                 url|            city|  price|cylinders_numeric|odometer|manufacturer|condition|  fuel|   type|title_status|transmission|  drive|      lat|      long|year|
+--------------------+----------------+-------+-----------------+--------+------------+---------+------+-------+------------+------------+-------+---------+----------+----+
|https://grandrapi...|grand rapids, MI| 8900.0|                6|119000.0|     lincoln|  Unknown|   gas|Unknown|       clean|   automatic|Unknown|  42.9737|  -85.7265|2009|
|https://grandrapi...|grand rapids, MI| 7995.0|                6|129105.0|    cadillac|  Unknown|   gas|Unknown|       clean|   automatic|Unknown|43.186723|-84.163862|2010|
|https://grandrapi...|grand rapids, MI| 6995.0|                6|164296.0|     Unknown|  Unknown|   gas|Unknown|       clean|   automat

In [19]:
def add_to_report(step, column_affected, log):
    """Internal helper to standardize report entries."""
    report = {
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'step': step,
        'column_affected/created': column_affected,
        'log': log
    }
    feature_engineering_logs.append(report)

In [20]:
from pyspark.sql import functions as F

condition_map = (F.when(F.col("condition") == "new", 5)
                  .when(F.col("condition") == "like new", 4)
                  .when(F.col("condition") == "excellent", 3)
                  .when(F.col("condition") == "good", 2)
                  .when(F.col("condition") == "fair", 1)
                  .otherwise(0))
log=["Ordinal encoding applied on condition"]
add_to_report("Encoding", "condition_rank", log)

title_map = (F.when(F.col("title_status") == "clean", 5)
               .when(F.col("title_status") == "lien", 4)
               .when(F.col("title_status") == "rebuilt", 3)
               .when(F.col("title_status") == "missing", 2)
               .when(F.col("title_status") == "salvage", 1)
               .otherwise(0))
log=["Ordinal encoding applied on title_status"]
add_to_report("Encoding", "title_status_rank", log)

df_engineered = df.withColumn("condition_rank", condition_map) \
                        .withColumn("title_status_rank", title_map) \
                        .withColumn("car_age", F.lit(2026) - F.col("year"))
log=["Created car_age from year, car_age = 2026 - year"]
add_to_report("Feature Interaction", "car_age", log)

In [21]:
df_engineered.select(
    F.min("car_age").alias("min_age"), 
    F.max("car_age").alias("max_age"),
    F.min("odometer").alias("min_odo"), 
    F.max("odometer").alias("max_odo")
).show()

+-------+-------+-------+--------+
|min_age|max_age|min_odo| max_odo|
+-------+-------+-------+--------+
|      6|    126|    0.0|272410.0|
+-------+-------+-------+--------+



In [22]:
age_splits = [float("-inf"), 10, 30, 50, 80, float("inf")]
odo_splits = [float("-inf"), 30000, 80000, 120000, 200000, float("inf")]

# 2. Apply Bucketizer for car_age
age_bucketizer = Bucketizer(splits=age_splits, inputCol="car_age", outputCol="car_age_bin")
df_engineered = age_bucketizer.setHandleInvalid("keep").transform(df_engineered)
log=["discretization and encoding applied on car_age"]
add_to_report("Binning and Encoding", "car_age_bin", log)

# 3. Apply Bucketizer for odometer
odo_bucketizer = Bucketizer(splits=odo_splits, inputCol="odometer", outputCol="odometer_bin")
df_engineered = odo_bucketizer.setHandleInvalid("keep").transform(df_engineered)
log=["discretization and encoding applied on odometer"]
add_to_report("Binning and Encoding", "odometer_bin", log)


# Show a quick check
df_engineered.select("car_age", "car_age_bin", "odometer", "odometer_bin").show(5)

+-------+-----------+--------+------------+
|car_age|car_age_bin|odometer|odometer_bin|
+-------+-----------+--------+------------+
|     17|        1.0|119000.0|         2.0|
|     16|        1.0|129105.0|         3.0|
|     19|        1.0|164296.0|         3.0|
|     16|        1.0|123213.0|         3.0|
|     16|        1.0|253000.0|         4.0|
+-------+-----------+--------+------------+
only showing top 5 rows


In [23]:
ohe_cols = ["fuel", "drive", "transmission"]
for col in ohe_cols:
    indexer = StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
    df_engineered = indexer.fit(df_engineered).transform(df_engineered)

    
    encoder = OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_ohe")
    df_engineered = encoder.fit(df_engineered).transform(df_engineered)

    log=[f"One hot encoding applied on {col}"]
    add_to_report("Encoding", f"{col}_ohe", log)

In [24]:
def apply_binary_encoding(df, col_name):
    # 1. Get unique IDs
    indexer = StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx")
    df = indexer.fit(df).transform(df)
    
    # 2. Calculate how many bits we need (e.g., 40 manufacturers need 6 bits because 2^6 = 64)
    max_id = int(df.agg(F.max(f"{col_name}_idx")).collect()[0][0])
    num_bits = math.ceil(math.log2(max_id + 1))
    
    # 3. Create a column for each bit using bitwise shifts
    bit_cols = []
    for i in range(num_bits):
        bit_name = f"{col_name}_bit_{i}"
        # (id >> i) & 1  extracts the i-th bit
        df = df.withColumn(bit_name, F.shiftRight(F.col(f"{col_name}_idx").cast("long"), i).bitwiseAND(1))
        bit_cols.append(bit_name)
    
    return df, bit_cols

# Apply it
df_engineered, manufacturer_bits = apply_binary_encoding(df_engineered, "manufacturer")
log=[f"Binary encoding applied on manufacturer"]
add_to_report("Encoding", "manufacturer_bits", log)

df_engineered, type_bits = apply_binary_encoding(df_engineered, "type")
log=[f"Binary encoding applied on type"]
add_to_report("Encoding", "type_bits", log)


c:\Users\lenovo\miniconda3\envs\spark_stable\Lib\site-packages\pyspark\sql\functions\builtin.py:7937: FutureWarning: Deprecated in 3.2, use shiftright instead.
  warnings.warn("Deprecated in 3.2, use shiftright instead.", FutureWarning)


In [25]:

# 1. Your specified list (corrected typo 'feul' -> 'fuel')
featured_cols = [
    "cylinders_numeric", "odometer_bin", "lat", "long", 
    "car_age_bin", "condition_rank", "title_status_rank", 
    "fuel_ohe", "transmission_ohe", "drive_ohe"
]

# 2. Combine with the binary bit columns we generated earlier
# 'manufacturer_bits' and 'type_bits' were the lists returned by our custom function
all_model_cols = featured_cols + manufacturer_bits + type_bits + ["price"]

# 3. Create the final features DataFrame
df_features = df_engineered.select(all_model_cols)

# Quick check on the structure
print(f"Total columns in feature set: {len(df_features.columns)}")
df_features.show(5, truncate=False)

Total columns in feature set: 21
+-----------------+------------+---------+----------+-----------+--------------+-----------------+-------------+----------------+-------------+------------------+------------------+------------------+------------------+------------------+------------------+----------+----------+----------+----------+-------+
|cylinders_numeric|odometer_bin|lat      |long      |car_age_bin|condition_rank|title_status_rank|fuel_ohe     |transmission_ohe|drive_ohe    |manufacturer_bit_0|manufacturer_bit_1|manufacturer_bit_2|manufacturer_bit_3|manufacturer_bit_4|manufacturer_bit_5|type_bit_0|type_bit_1|type_bit_2|type_bit_3|price  |
+-----------------+------------+---------+----------+-----------+--------------+-----------------+-------------+----------------+-------------+------------------+------------------+------------------+------------------+------------------+------------------+----------+----------+----------+----------+-------+
|6                |2.0         |42.97

In [26]:
def expand_ohe(df, ohe_col):
    # 1. Convert Vector to Array
    df = df.withColumn(f"{ohe_col}_arr", vector_to_array(F.col(ohe_col)))
    
    # 2. Get the size of the OHE vector from the schema (no action triggered!)
    # This looks at the column metadata rather than the actual data
    field = df.schema[f"{ohe_col}_arr"]
    # If for some reason we can't get it from schema, we'll do a one-time check
    try:
        num_elements = df.select(F.size(f"{ohe_col}_arr")).first()[0]
    except:
        return df # Fallback if empty
    
    # 3. Use a list comprehension to create new columns at once
    # This is much faster for Spark's optimizer than a loop of .withColumn
    cols_to_add = [F.col(f"{ohe_col}_arr")[i].alias(f"{ohe_col}_{i}") for i in range(num_elements)]
    
    return df.select("*", *cols_to_add).drop(ohe_col, f"{ohe_col}_arr")

# Apply it
df_expanded = df_features
for col in ["fuel_ohe", "transmission_ohe", "drive_ohe"]:
    df_expanded = expand_ohe(df_expanded, col)
    log=[f"Expanding one hot encoding column of {col}"]
    add_to_report("Expanding", f"{col}", log)
print("✅ OHE Vectors successfully expanded into normal columns!")
# 2. Define the final list of columns to be scaled
# This includes original numerics, bins, ranks, and the new expanded OHE bits + binary bits
all_cols_to_scale = ["cylinders_numeric", "lat", "long"]
all_cols = [
    "cylinders_numeric", "lat", "long", "car_age_bin", 
    "odometer_bin", "condition_rank", "title_status_rank"
] + [c for c in df_expanded.columns if "_ohe_" in c] + manufacturer_bits + type_bits

✅ OHE Vectors successfully expanded into normal columns!


In [27]:
# 1. Split BEFORE scaling to avoid data leakage (60/20/20)
df_expanded.toPandas().to_csv(f"data/processed/full_dataset.csv", index=False)
train_df, val_df, test_df = df_expanded.randomSplit([0.6, 0.2, 0.2], seed=42)

print(f"📈 Training set:   {train_df.count()} rows")
print(f"🧪 Validation set: {val_df.count()} rows")
print(f"🏁 Test set:       {test_df.count()} rows")

📈 Training set:   249738 rows
🧪 Validation set: 82709 rows
🏁 Test set:       82707 rows


In [29]:
# 2. Calculate Mean and StdDev ONLY from the training set
stats = train_df.select([F.mean(c).alias(f"{c}_mean") for c in all_cols_to_scale] + 
                         [F.stddev(c).alias(f"{c}_std") for c in all_cols_to_scale]).collect()[0]

# 3. Apply the scaling formula to all three sets
def manual_scaler(df, stats_row, cols, cols_to_scale):
    for c in cols_to_scale:
        mean_val = stats_row[f"{c}_mean"]
        std_val = stats_row[f"{c}_std"]
        # Avoid division by zero if std is 0
        if std_val == 0 or std_val is None:
            df = df.withColumn(c, F.lit(0.0))
        else:
            df = df.withColumn(c, (F.col(c) - mean_val) / std_val)
    return df.select(cols + ["price"])

train_scaled = manual_scaler(train_df, stats, all_cols, all_cols_to_scale)
val_scaled = manual_scaler(val_df, stats, all_cols, all_cols_to_scale)
test_scaled = manual_scaler(test_df, stats, all_cols, all_cols_to_scale)
full_scaled = manual_scaler(df_expanded, stats, all_cols, all_cols_to_scale)

In [30]:
print("💾 Saving separate column CSVs...")

# Helper to save via Pandas
def save_csv(df, name):
    df.toPandas().to_csv(f"data/processed/{name}.csv", index=False)
    print(f" - {name}.csv saved.")

save_csv(full_scaled, "full_dataset_scaled")
save_csv(train_scaled, "train_scaled")
save_csv(val_scaled, "val_scaled")
save_csv(test_scaled, "test_scaled")

print("✅ Finished! All features are scaled and saved as individual columns.")

💾 Saving separate column CSVs...
 - full_dataset_scaled.csv saved.
 - train_scaled.csv saved.
 - val_scaled.csv saved.
 - test_scaled.csv saved.
✅ Finished! All features are scaled and saved as individual columns.


In [31]:
def generate_report(json_path="feature_engineering_report.json"):
        """Saves cleaning results to JSON and CSV and returns summary statistics."""
        
        report_data = {
            "logs": feature_engineering_logs,
        }

        with open(json_path, 'w') as f:
            json.dump(report_data, f, indent=4)


        
        print(f"\n{'='*55}\n Cleaning COMPLETE\n{'='*55}")
        print(f"Files Saved: {json_path}")

        return report_data

generate_report("Reports/Results/feature_engineering_report.json")


 Cleaning COMPLETE
Files Saved: Reports/Results/feature_engineering_report.json


{'logs': [{'timestamp': '2026-05-01 21:10:58',
   'step': 'Encoding',
   'column_affected/created': 'condition_rank',
   'log': ['Ordinal encoding applied on condition']},
  {'timestamp': '2026-05-01 21:10:58',
   'step': 'Encoding',
   'column_affected/created': 'title_status_rank',
   'log': ['Ordinal encoding applied on title_status']},
  {'timestamp': '2026-05-01 21:10:58',
   'step': 'Feature Interaction',
   'column_affected/created': 'car_age',
   'log': ['Created car_age from year, car_age = 2026 - year']},
  {'timestamp': '2026-05-01 21:10:58',
   'step': 'Binning and Encoding',
   'column_affected/created': 'car_age_bin',
   'log': ['discretization and encoding applied on car_age']},
  {'timestamp': '2026-05-01 21:10:58',
   'step': 'Binning and Encoding',
   'column_affected/created': 'odometer_bin',
   'log': ['discretization and encoding applied on odometer']},
  {'timestamp': '2026-05-01 21:10:59',
   'step': 'Encoding',
   'column_affected/created': 'fuel_ohe',
   'log':